### Imports necessários

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('01-Core/'))

In [ ]:
import numpy as np
from matplotlib import pyplot
from Star.Estrela import Estrela, GeometriaCME
from Planet.Eclipse import Eclipse
from Planet.Planeta import Planeta
from Planet.Moon import Moon
from Misc.Verify  import Validar,calSemiEixo,calculaLat
import cv2 as cv
import numpy as np

%matplotlib tk

## Modelando a Estrela

In [5]:
### Simulação com limbo correto
parametros_modelo = {
    'rsun': 0.805,
    'u1': 0.495,
    'u2': 0.0912,
    'raio_plan_Jup': 1.138,
    'semi_eixo_UA': 0.031,
    'angulo_inclinacao': 85.710,
    'periodo': 2.21857520,
    # Parâmetro medido dos dados para alinhamento
    't0': 1 # Centro do trânsito default
}

### Simulação com limbo genérico que representa a atmosfera do planeta
parametros_modelo = {
    'rsun': 0.805,
    'u1': 1,
    'u2': 1,
    'raio_plan_Jup': 0.95,
    'semi_eixo_UA': 0.0401,
    'angulo_inclinacao': 88.526,
    'periodo': 2.31857520,
    'mass_planeta': 1.138,
    # Parâmetro medido dos dados para alinhamento
    't0': 1 # Centro do trânsito default
}

# modelo sinal obs 06
parametros_modelo = {
    'rsun': 0.805,
    'u1': 1,
    'u2': 1,
    'raio_plan_Jup': 1.05, #ou 0.95
    'semi_eixo_UA': 0.0401,
    'angulo_inclinacao': 88.526,
    'periodo': 2.31857520,
    'mass_planeta': 1.138,
    # Parâmetro medido dos dados para alinhamento
    'tC': 1 # Centro do trânsito default
}

use_fits = False # mude caso queira usar o modelo de estrela com fits
raio_estrela_pixel = 373. # default (pixel)
intensidade_maxima = 240 # default
tamanho_matriz = 856 # default
raio_estrela = 0.805 # raio da estrela em relacao ao raio do sol
coeficiente_um = 1
coeficiente_dois = 1

#cria estrela
estrela_ = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits=use_fits, fits_path="2011-06-05")
tamanho_matriz = estrela_.getTamanhoMatriz()

Nx = estrela_.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_.getNy()
dtor = np.pi/180.  

## Modelando o Planeta

In [6]:
periodo = 2.21857520# em dias
angulo_inclinacao = 88.526 # em graus
ecc = 0 # excentricidade
anomalia = 0 # anomalia
raio_plan_Jup = 1.05 # em relação ao raio de jupiter
semi_eixo_UA = 0.0401 # UA
mass_planeta = 1.138 #em relacao ao R de jupiter

planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_.getRaioSun(), mass_planeta)

print(planeta_.getRaioPlan())


0.13095364458391764


In [7]:
estrela_matriz = estrela_.getMatrizEstrela()
estrela_.Plotar(tamanho_matriz, estrela_matriz)

In [8]:
#eclipse
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_, planeta_)
estrela_.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

#Plotagem da curva de luz 
pyplot.plot(tempoHoras, curvaLuz)
pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz)-0.001, 1.005])                       
pyplot.show()



Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 4.664877350855445


In [6]:
latsugerida = eclipse_.calculaLatMancha()

A latitude sugerida para que a mancha influencie na curva de luz da estrela é: -15.683224806877336


invalid command name "140308588977024_on_timer"
    while executing
"140308588977024_on_timer"
    ("after" script)


# Modelando interferências na curva de Luz

## Adicionando manchas
***

In [7]:
raio_mancha = 0.20
intensidade = 0.5 
latitude = -15
longitude = 5

raio_mancha2 = 0.30
intensidade2 = 0.5 
latitude2 = -15
longitude2 = 8

mancha = Estrela.Mancha(intensidade, raio_mancha, latitude, longitude)
mancha2 = Estrela.Mancha(intensidade2, raio_mancha2, latitude2, longitude2) 


estrela_.addMancha(mancha)
estrela_.addMancha(mancha2) #caso queira adicionar mais manchas

estrela_.criaEstrelaManchada()
estrela_matriz = estrela_.getMatrizEstrela()

### Visualizar Eclipse com Manchas

In [241]:
# Eclipse
cme = False 

# Passa para o eclipse a estrela atualizada
eclipse_.setEstrela(estrela_matriz)

estrela_.Plotar(tamanho_matriz, estrela_matriz)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):",eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

#Plotagem da curva de luz 
pyplot.plot(tempoHoras,curvaLuz)
pyplot.axis([-tempoTransito/2,tempoTransito/2,min(curvaLuz)-0.001,1.005])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 4.875141909426886


## Adicionando fáculas
***

In [13]:
raio_fácula = 0.25
intensidade_facula = 1.6
latitude_facula = -39
longitude_facula = -28


facula = Estrela.Facula(intensidade_facula, raio_fácula, latitude_facula, longitude_facula)
#facula2 = Estrela.Facula(1.5, 0.08, latitude_facula, longitude_facula) #caso queira adicionar mais fáculas


estrela_.addFacula(facula)
#estrela_.addFacula(facula2)

estrela_.criaEstrelaComFaculas()
estrela_matriz = estrela_.getMatrizEstrela()

In [14]:
# Eclipse

# Passa para o eclipse a estrela atualizada
eclipse_.setEstrela(estrela_matriz)

estrela_.Plotar(tamanho_matriz, estrela_matriz)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):",eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

#Plotagem da curva de luz 
pyplot.plot(tempoHoras,curvaLuz)
pyplot.axis([-tempoTransito/2,tempoTransito/2,min(curvaLuz)-0.001,1.005])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 4.978435644173519


## Adicionando luas
***

In [9]:
rmoon = 0.3 #em relacao ao raio da Terra
mass = 0.5 #em relacao a massa da Terra
perLua = 0.03 #em dias 

moon = Moon(rmoon, 
            mass,
            perLua,
            tempoHoras,
            planeta_.anguloInclinacao,
            planeta_.mass,
            planeta_.getRaioPlanPixel(estrela_.raio, estrela_.raioSun),
            estrela_.raio,
            estrela_.getRaioSun(),
            planeta_.periodo)

eclipse_.criarLua(moon) #adiciona lua no planeta que esta no eclipse

# Criando planeta com lua 
estrela_matriz = estrela_.getMatrizEstrela()

# Passa para o eclipse a estrela atualizada
eclipse_.setEstrela(estrela_matriz)

### Analisando curva de luz

In [10]:
#eclipse
moon.setMoonName("lua teste 01")
estrela_.Plotar(tamanho_matriz, estrela_matriz)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

#Plotagem da curva de luz 
pyplot.plot(tempoHoras,curvaLuz)
pyplot.axis([-tempoTransito/2,tempoTransito/2,min(curvaLuz)-0.001,1.005])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

LUA::::: lua teste 01
Tempo Total (Trânsito): 4.664877350855445


## Adicionando CMEs
***

_A análise à seguir é realizada apenas a fins de demonstração e visualização, já que CMEs são observadas apenas em comprimentos de onda UV e EUV_

### CME modelo genérico para testes

In [11]:
estrela_matriz = estrela_.getMatrizEstrela()

intensidade_cme = estrela_.temperaturaEfetiva-800 # intensidade calculada em relação à intensidade da estrela
raio_cme = 120 # em Pixel
velocidade_cme = 0.188 # UA/h 
altura_inicial_cme = -1 # Em UA (valores negativos muito altos servem para simular a CME não alcançando a atmosfera do planeta)
opacidade = 0.515
taxa_escurecimento = 15
p0x = 372
p1y = 250
p1x = 272
p0y = 90

cme = Estrela.EjecaoMassa(raio_cme, p0x, p0y, p1x, p1y, opacidade, intensidade_cme, velocidade_cme, taxa_escurecimento, altura_inicial_cme, GeometriaCME.PROJECAO_DISCO) # mude a projeção conforme desejado

estrela_.addCme(cme)

### CME projeção de disco

In [ ]:
estrel_amatriz = estrela_.getMatrizEstrela()

# chute inicial capsula
intensidade_cme = estrela_.temperaturaEfetiva-800 # temperatura da CME (em K) 4875.0
raio_cme = 110 # pixel
velocidade_cme = 0.152506053839219 # UA/h 
altura_inicial_cme = -1 # Em UA (valores negativos muito altos servem para simular a CME não alcançando a atmosfera do planeta)
opacidade = 0.3289613239247936
taxa_escurecimento = 120.01068829123473
p0x = 291
p0y = 200
p1x = 201
p1y = 271

cme = Estrela.EjecaoMassa(raio_cme, p0x, p0y, p1x, p1y, opacidade, intensidade_cme, velocidade_cme, taxa_escurecimento, altura_inicial_cme, GeometriaCME.PROJECAO_DISCO)

estrela_.addCme(cme)

### CME projeção perfil (gota) testes com maior massa

In [ ]:
estrel_amatriz = estrela_.getMatrizEstrela()

### chute inicial gota maior massa
intensidade_cme = estrela_.temperaturaEfetiva-800 # temperatura da CME (em K) 4875.0
raio_cme = 95 # em ??
velocidade_cme = 0.188 # UA/h 
altura_inicial_cme = -1 # Em UA (valores negativos muito altos servem para simular a CME não alcançando a atmosfera do planeta)
opacidade = 0.815
taxa_escurecimento = 49.596847171666624
p0x = 561
p0y = 540
p1x = 565
p1y = 350

cme = Estrela.EjecaoMassa(raio_cme, p0x, p0y, p1x, p1y, opacidade, intensidade_cme, velocidade_cme, taxa_escurecimento, altura_inicial_cme, GeometriaCME.VISAO_PERFIL)

estrela_.addCme(cme)

### CME projeção perfil (gota) testes com menor massa

In [ ]:
estrel_amatriz = estrela_.getMatrizEstrela()

### chute inicial gota menor massa
intensidade_cme = estrela_.temperaturaEfetiva-800 # temperatura da CME (em K) 4875.0
raio_cme = 90 # pixel
velocidade_cme = 0.15327006832062182 # UA/h 
altura_inicial_cme = -1 # Em UA (valores negativos muito altos servem para simular a CME não alcançando a atmosfera do planeta)
opacidade = 0.49207760243462
taxa_escurecimento = 149.97604707164706
p0x = 351
p0y = 220
p1x = 191
p1y = 291

cme = Estrela.EjecaoMassa(raio_cme, p0x, p0y, p1x, p1y, opacidade, intensidade_cme, velocidade_cme, taxa_escurecimento, altura_inicial_cme, GeometriaCME.VISAO_PERFIL)

estrela_.addCme(cme)

### Visualização

In [12]:
#eclipse
eclipse_.setEstrela(estrela_matriz)
eclipse_.cmap = "copper"
estrela_.Plotar(tamanho_matriz, estrela_matriz)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):",eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

#Plotagem da curva de luz 
pyplot.plot(tempoHoras,curvaLuz)
pyplot.axis([-tempoTransito/2,tempoTransito/2,min(curvaLuz)-0.001,1.015])                       
pyplot.show()

estrela_.get_area_cme_km2()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 4.664877350855445
Área da CME relativa à Estrela: 20.71 %
Área da CME: 204,442,476,709.14 km²


204442476709.13904